# 9. The grand tour: one mission architecture, every pane

`analysis.grand_dashboard` puts the analysis views of tutorials 3 to 8 on one
linked dashboard. Every pane of the dashboard follows one **subject**.

The subject is one **mission architecture**. A mission architecture is one
mission definition, one airframe, and one variant for each other variation
point. A **variation point** is a feature that declares a closed list of
variants, with the SysML keywords `variation` and `variant`. The mission
definitions `IsrUav`, `LogisticsUav` and `InterceptUav` declare the variation
points of the DeepScout fleet.

The dashboard has three controls:

- The mission selector sets the mission definition.
- A click on a craft in the containment tree or in the structure diagram sets
  the airframe.
- The knobs set the other variation points and the bounded mission attributes.

When the subject changes, every pane shows the new subject. Each reaction is a
traitlets observer in the kernel. Thus the cells below prove each reaction
without a browser. The dashboard needs the extras `viz`, `mdao`, `smt` and
`diagrams`.

In [ ]:
import json

import longeron
from longeron.analysis.grand import grand_dashboard

program = longeron.load("../examples/deepscout")  # the whole DeepScout program: one workspace

## Build the dashboard

One call builds the dashboard. It has four rows:

1. A header that names the subject and shows its requirements score.
2. The containment tree, the structure diagram, the 3D pane and the
   requirements scoreboard.
3. The subject card, the camera card, the OpenMDAO sizing strip and the Z3
   verdict strip.
4. The Cesium pane, which flies the subject over a satellite map.

The containment tree lists every package, definition and usage of the model by
ownership. Tutorial 3 shows the tree and the diagram.

The default subject is the winner of the ISR mission. The ISR mission
(`IsrUav`) holds a sensor on station as long as possible. Its metric is
`stationMinutes`. For each airframe, the trade study of tutorial 4 supplies a
**preset**. A preset is the best selection of the other variation points for
one airframe in one mission. The winner is the airframe whose preset has the
best metric.

In [ ]:
dash = grand_dashboard(program)  # mission="ISR", airframe=None: the mission's winner

assert dash.subject.mission == "ISR" and dash.subject.airframe == "flyingWingSingle"
assert dash.craft == "FlyingWings::FlyingWingSingle"
assert round(dash.architecture.metrics["stationMinutes"], 1) == 274.6
assert dash.architecture.violations == []  # the preset satisfies every assert constraint
print(dash.subject)

## The knobs come from the model

A **knob** is a slider or a dropdown that the dashboard derives from the model
for the subject. No Python list declares the knobs. The rule has two parts:

- Each variation point other than `airframe` becomes a dropdown over its
  variants.
- Each input attribute that constraints bound on both sides becomes a slider
  over that interval.

An input attribute is an attribute whose value is a literal, not a derived
expression. The ISR mission declares `minStationMinutes`, the least station
time that the tasking demands. Two `assert constraint` bounds hold it in
[25, 240]. The airframe `flyingWingSingle` bounds its span, its mean chord
and its loiter speed the same way. Thus the ISR subject gets four sliders. An
airframe slider has the name `airframe.<attribute>`.

A knob change is a **transient override**. The analysis uses the new value,
but the model keeps its declared value. The inspector of tutorial 3 edits the
model.

In [ ]:
knobs = dash.knob_set
assert [d.point for d in knobs.dropdowns] == ["sensor", "motors", "props", "battery", "material"]
assert [(s.name, s.low, s.high) for s in knobs.sliders] == [
    ("minStationMinutes", 25.0, 240.0),
    ("airframe.wingSpan", 1.6, 2.8),
    ("airframe.wingChord", 0.26, 0.4),
    ("airframe.loiterSpeed", 11.5, 19.0),
]

# a dropdown sets one variation point: the 3D pane re-bakes, the other points keep their preset
mesh = dash.viewer.mesh_json
dash.knobs["battery"].value = "tattu3s"
assert dash.viewer.mesh_json != mesh and dash.subject.selection["motors"] == "mn4006"
print("tattu3s:", round(dash.architecture.metrics["stationMinutes"], 1), "station minutes")

# a slider sets a transient override: the trade study sees 240, the model keeps 25
dash.knobs["minStationMinutes"].value = 240.0
assert dash.subject.overrides == {"minStationMinutes": 240.0}
assert "stationReq" in dash.architecture.violations
assert "stationReq" not in dash.studies["ISR"].evaluate(dash.subject.mix()).violations

## The scoreboard scores the mission's requirement set

The scoreboard of tutorial 6 scores the **requirement set** of the subject.
The requirement set holds each requirement whose subject type the mission
architecture or its airframe conforms to. Each requirement scores for the
instance where it applies. `dash.requirements` maps each requirement to that
instance.

`IsrTasking` compares the station time of the mission architecture with
`minStationMinutes`. A flying wing holds station in loiter flight, without
hover. Thus `IsrTasking` applies to the ISR winner, and the scoreboard measures it.
After the two knob changes above, the station time is less than
`minStationMinutes`.

In [ ]:
def leaf(node, name):
    if node["label"] == name:
        return node
    found = (leaf(child, name) for child in node.get("children", ()))
    return next((hit for hit in found if hit), None)


def utility(name):
    return leaf(json.loads(dash.board.nodes_json), name)["utility"]


tasking = dash.requirements["ScoutMissions::MissionRequirements::IsrTasking"]
assert tasking.definition.qualified_name == "ScoutMissions::IsrUav"  # scored for the architecture
print(sorted(qname.rsplit("::", 1)[-1] for qname in dash.requirements))
assert utility("IsrTasking") == 0.0  # 43 station minutes on tattu3s, floor 240

dash.knobs["battery"].value = "liion6s6p"
dash.knobs["minStationMinutes"].value = 25.0
assert utility("IsrTasking") == 1.0  # the winner's preset holds station

## The camera what-if measures the subject's camera

The camera card turns the boresight of the camera that the airframe of the
subject declares. The analysis case `ScoutSurfaces::CameraWhatIf` bounds the
two sliders with its assume constraints. Each move runs the occlusion measure
of tutorial 7 on the mesh of the subject again.

The requirement `clearView` permits no airframe part in the view cone. Its
require constraint is `occludedFraction <= 0.0`. The scoreboard measures
`clearView` for each subject at the declared boresight. If the boresight looks
aft through the airframe, parts of the craft enter the view cone. The
interpreter then reports that `clearView` fails, and the scoreboard shows the
loss.

In [ ]:
clear_view = "DeepScout::installation::clearView"
oracle = dash.studies["ISR"].interp  # the interpreter checks the requirement's own constraint

assert (dash.elevation.min, dash.elevation.max) == (-90.0, 90.0)  # from CameraWhatIf
assert dash.camera["elevation"] == dash.elevation.value  # the declared boresight
assert leaf(json.loads(dash.board.nodes_json), "clearView")["measured"]
occluded = dash.report["occludedFraction"]
assert occluded == 0.0 and utility("clearView") == 1.0  # the nose camera sees no airframe
assert oracle.check_requirement(clear_view, occludedFraction=occluded).satisfied
print(f"occludedFraction {occluded:.0e} at the declared boresight")

dash.azimuth.value = 180.0  # look aft through the airframe
occluded = dash.report["occludedFraction"]
assert occluded > 0.0 and utility("clearView") < 1.0
assert not oracle.check_requirement(clear_view, occludedFraction=occluded).satisfied
print("aft:", sorted(dash.report["obstructions"]))
dash.azimuth.value = dash.camera["azimuth"]

## Sizing and Z3 follow the mission

A **sizing context** is a sizing part that the model declares for one mission.
It freezes one component mix and keeps the loiter speed continuous. The ISR
context is `ScoutSizing::IsrPrime`, a frozen `vtolWing` mix. The sizing
context does not follow the airframe of the subject.

The ISR context drives two strips. Its OpenMDAO problem (tutorial 4) drives
the sizing strip. Its station requirement `ScoutSizing::IsrStation` drives the
Z3 verdict strip (tutorial 6). The assert constraints of `IsrPrime` bound the
loiter slider and the optimizer.

The model declares no sizing context for logistics or for intercept. For
those missions, both strips show an **absence card**. An absence card states
that the model declares no sizing context for the mission.

The *maximize station time* button runs the OpenMDAO **driver**. The driver
is the optimizer that moves the design variable `loiterSpeed` to the largest
`stationMinutes`. A **driver iteration** is one model evaluation that the
driver records. After the run, the sizing strip shows three renderings:

- The history plot shows `stationMinutes` and each constraint **margin** for
  each driver iteration. A margin is zero at the bound of its constraint and
  positive while the constraint holds.
- The table lists the design variable and the constraints with their bounds
  and current values. A constraint is **active** when its value is equal to a
  bound at the optimum. At the ISR optimum, the stall floor `aboveStall` is
  active, so the slider stops at 11 m/s.
- The **N2 matrix** shows the structure of the OpenMDAO problem, in the layout
  of the OpenMDAO N2 diagram. Each component of the problem is on the
  diagonal. A mark in row *i* and column *j* shows that an output of component
  *i* feeds an input of component *j*. The hover title of a mark names the
  connected variables.

In [ ]:
from IPython.display import HTML

dash.loiter.value = 16.0
dash.optimize.click()  # the driver starts at 16 m/s
assert dash.loiter.value == 11.0  # and stops at the stall floor
assert "driver iterations" in dash.sizing_history.value
assert "aboveStall: margin +0.00" in dash.sizing_history.value  # the active bound
# the drag feeds the cruise power: one mark of the N2 matrix
assert "Aerodynamics.dragArea_comp -&gt; Propulsion.loiterPowerW_comp" in dash.sizing_n2.value
cards = (dash.sizing_history, dash.sizing_table, dash.sizing_n2)
HTML(
    '<div style="display:flex; gap:24px">'
    + "".join(f'<div style="width:240px">{card.value}</div>' for card in cards)
    + "</div>"
)

In [ ]:
assert (dash.loiter.min, dash.loiter.max) == (11.0, 24.0)  # IsrPrime's aboveStall, belowCruise
dash.loiter.value = 21.0
at_21 = float(dash.problem.problem.get_val("stationMinutes")[0])
dash.optimize.click()  # the driver runs, and the slider moves to the optimum
assert float(dash.problem.problem.get_val("stationMinutes")[0]) > at_21

# Z3: consistent at the design point, and the impossible what-if names its blockers
assert dash.smt_sat.status == "sat" and dash.smt_what_if.status == "unsat"
assert "IsrPrime::aboveStall" in dash.smt_what_if.core
dash.loiter.value = 15.0

dash.mission_toggle.value = "logistics"  # the airframe stays, its logistics preset fills the rest
assert dash.subject.airframe == "flyingWingSingle" and "cargo" in dash.subject.selection
assert dash.loiter is None and dash.smt_sat is None
assert "no context for logistics" in dash.sizing_title.value
print(sorted(dash.knobs))

## Click a craft to change the subject

A **catalog craft** is a variant of the `airframe` variation point, or the
definition that types the variant. A click on a catalog craft makes it the
airframe of the subject. The preset of the craft then fills the other
variation points. The tree and the diagram share one selection. A tree row
selects its diagram node, and a diagram node selects its tree row and reveals
it.

A **visiting craft** is a craft that the 3D pane shows but that is not the
subject. The five build configurations, for example `Rotorcraft::QuadCopter`,
are complete crafts but not catalog crafts. A click on a build configuration
shows it as a visiting craft, and the subject does not change.

In [ ]:
# a tree row sets the logistics winner: tree -> diagram -> subject
flight = dash.mission.czml_json
dash.tree.selected = ["ScoutMissions::Catalog::AirframeChoice::flyingWingTwinTip"]
assert dash.subject.airframe == "flyingWingTwinTip"
assert dash.config_view.current == dash.craft == "FlyingWings::FlyingWingTwinTip"
assert round(dash.architecture.metrics["payloadRangeKgKm"], 1) == 187.0
assert dash.mission.czml_json != flight  # the Cesium pane flies the new subject

# a diagram click sets the intercept winner, and the tree row follows
dash.mission_toggle.value = "intercept"
dash.diagram.view.selection.ids = ["ScoutMissions::Catalog::AirframeChoice::dartInterceptor"]
assert list(dash.tree.selected) == ["ScoutMissions::Catalog::AirframeChoice::dartInterceptor"]
assert round(dash.architecture.metrics["maxTargetSpeed"], 1) == 66.9

# a build configuration only visits
subject = dash.subject
dash.diagram.view.selection.ids = ["Rotorcraft::QuadCopter"]
assert dash.config_view.current == "Rotorcraft::QuadCopter" and dash.subject == subject
assert "The subject did not change." in dash.header.value

# back to the ISR winner for the live view
dash.mission_toggle.value = "ISR"
dash.diagram.view.selection.ids = ["ScoutMissions::Catalog::AirframeChoice::flyingWingSingle"]
dash.diagram.view.selection.ids = []
assert dash.subject.airframe == "flyingWingSingle" and dash.subject.overrides == {}
print("grand tour: every pane follows the subject")

## The dashboard

Try these steps in JupyterLab:

1. Click `hexLifter` under `AirframeChoice` in the containment tree. The 3D
   pane and the Cesium pane show `hexLifter`, and the scoreboard scores the
   new subject.
2. Set `battery` to `tattu3s` and look at the scoreboard.
3. Drag `azimuth` to 180 and look at `clearView`.
4. Click *maximize station time* in the sizing strip.
5. Select `logistics`. The sizing strip and the Z3 verdict strip show the
   absence card.
6. Press play on the Cesium timeline.

In [ ]:
dash